In [1]:
#Nake: Joshua Trabka
#Date: 6/5/2026
#Description: Creates a dashboard using dash and plotly for the AAC Outcomes Database




!pip install dash

In [4]:
import dash
from dash import dcc, html, dash_table, Input, Output
import plotly.express as px
import pandas as pd
from animal_shelter_CRUD import AnimalShelterCRUD

# 1. Initialize Dash Framework
app = dash.Dash(__name__, title="AAC Shelter Analytics")

# 2. Connect to the CRUD Service Layer
db = AnimalShelterCRUD(user="root", password="Xy25!Jb4g86#")

# 3. Define the Application Layout (HTML/CSS Component Tree)
app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'padding': '20px', 'backgroundColor': '#f9f9f9'}, children=[
    
    # Header Banner
    html.Div(style={'backgroundColor': '#1e293b', 'color': 'white', 'padding': '20px', 'borderRadius': '8px', 'marginBottom': '20px'}, children=[
        html.H1("🐾 Austin Animal Center Outcomes Dashboard", style={'margin': '0'}),
        html.P("Relational Database Interface powered by Dash & Plotly", style={'margin': '5px 0 0 0', 'opacity': '0.8'})
    ]),
    
    # Sidebar Control Panel & Summary Cards Row
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '20px'}, children=[
        
        # Control Dropdown Panel
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}, children=[
            html.Label("Select Animal Type Filter:", style={'fontWeight': 'bold', 'color': '#334155'}),
            dcc.Dropdown(
                id='animal-type-dropdown',
                options=[{'label': t, 'value': t} for t in ["All", "Dog", "Cat", "Bird", "Livestock", "Other"]],
                value='All',
                clearable=False,
                style={'marginTop': '8px'}
            )
        ]),
        
        # Summary Card 1: Record Volume
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)', 'textAlign': 'center'}, children=[
            html.H3("Outcomes Evaluated", style={'margin': '0', 'color': '#64748b', 'fontSize': '14px'}),
            html.H1(id='metric-total-records', style={'margin': '10px 0 0 0', 'color': '#0f172a'})
        ]),
        
        # Summary Card 2: Breed Count
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)', 'textAlign': 'center'}, children=[
            html.H3("Unique Breeds Represented", style={'margin': '0', 'color': '#64748b', 'fontSize': '14px'}),
            html.H1(id='metric-unique-breeds', style={'margin': '10px 0 0 0', 'color': '#2563eb'})
        ])
    ]),
    
    # Charts Layer (Two Column Layout)
    html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '20px'}, children=[
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}, children=[
            html.H4("Top 10 Most Common Breeds", style={'marginTop': '0', 'color': '#334155'}),
            dcc.Graph(id='breed-bar-chart')
        ]),
        html.Div(style={'flex': '1', 'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}, children=[
            html.H4("Outcome Profile Distribution", style={'marginTop': '0', 'color': '#334155'}),
            dcc.Graph(id='outcome-pie-chart')
        ])
    ]),
    
    # Map Layer
    html.Div(style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)', 'marginBottom': '20px'}, children=[
        html.H4("Geospatial Outcome Coordinate Map", style={'marginTop': '0', 'color': '#334155'}),
        dcc.Graph(id='location-mapbox')
    ]),
    
    # Tabular Grid Layer
    html.Div(style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '8px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}, children=[
        html.H4("Active Relational Engine Grid View (Latest Logs)", style={'marginTop': '0', 'color': '#334155'}),
dash_table.DataTable(
    id='database-table',
    page_size=15,
    sort_action='native',          # Allows clicking column headers to sort ascending/descending
    filter_action='native',        # Adds an interactive text-input row at the top of each column
    filter_options={
        'case': 'insensitive',     # Makes typing "dog", "Dog", or "DOG" match identically
        'placeholder_text': 'Filter column...'
    },
    
    # Existing presentation styling
    style_table={'overflowX': 'auto'},
    style_header={'backgroundColor': '#1e293b', 'fontWeight': 'bold', 'color': 'white'},
    style_cell={'textAlign': 'left', 'padding': '10px', 'fontSize': '13px'},
    style_data_conditional=[
        {'if': {'row_index': 'odd'}, 'backgroundColor': '#f8fafc'}
    ]
)
    ])
])

# 4. Reactive Reactive Callbacks (The Link Between Inputs and Outputs)
@app.callback(
    [Output('metric-total-records', 'children'),
     Output('metric-unique-breeds', 'children'),
     Output('breed-bar-chart', 'figure'),
     Output('outcome-pie-chart', 'figure'),
     Output('location-mapbox', 'figure'),
     Output('database-table', 'data'),
     Output('database-table', 'columns')],
    [Input('animal-type-dropdown', 'value')]
)
def update_dashboard_metrics(selected_type):
    # Route input values cleanly through our predefined CRUD module
    filter_val = None if selected_type == "All" else selected_type
    df = db.read_shelter_data(animal_type_filter=filter_val, limit=15000)
    
    if df.empty:
        return "0", "0", {}, {}, {}, [], []
    
    # Compute Core KPIs
    total_logs = f"{len(df):,}"
    total_breeds = f"{df['breed'].nunique():,}"
    
    # Graph A: Top 10 Breeds Bar Chart
    breed_data = df['breed'].value_counts().head(10).reset_index()
    breed_data.columns = ['Breed', 'Count']
    fig_breed = px.bar(breed_data, x='Count', y='Breed', orientation='h', 
                       color='Count', color_continuous_scale='Blues')
    fig_breed.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False, margin=dict(l=20, r=20, t=20, b=20))
    
    # Graph B: Outcome Distribution Pie Chart
    outcome_data = df['outcome_type'].value_counts().reset_index()
    outcome_data.columns = ['Outcome Type', 'Count']
    fig_outcome = px.pie(outcome_data, values='Count', names='Outcome Type', hole=0.4,
                         color_discrete_sequence=px.colors.qualitative.Safe)
    fig_outcome.update_layout(margin=dict(l=20, r=20, t=20, b=20))
    
    # Graph C: Mapbox OpenStreet Maps
    map_df = df.dropna(subset=['location_lat', 'location_long'])
    if not map_df.empty:
        fig_map = px.scatter_mapbox(
            map_df, lat='location_lat', lon='location_long',
            hover_name='name', hover_data=['animal_type', 'breed', 'outcome_type'],
            zoom=10, height=400
        )
        fig_map.update_layout(mapbox_style="open-street-map", margin=dict(l=0, r=0, t=0, b=0))
    else:
        fig_map = {}

    # Build Table Columns and Data
    table_columns = [{"name": i, "id": i} for i in df.columns]
    table_data = df.to_dict('records')
    
    return total_logs, total_breeds, fig_breed, fig_outcome, fig_map, table_data, table_columns


# 5. Boot Up Application Server
if __name__ == '__main__':
    app.run(debug=True, port=8050)

2026-06-05 13:05:43,694 - INFO - Database CRUD Module initialized successfully.
